In [1]:
import csv
import numpy as np
import pandas as pd
import sys, os
import re
import ast
import datetime as dt
from tqdm import tqdm
from collections import defaultdict, Counter

import warnings

# 禁用所有警告
warnings.filterwarnings("ignore")

In [2]:
def median_lab_process(ts):
    # 将所有列值全部为 0 的列替换为 NaN
    ts.loc[:, (ts == 0).all(axis=0)] = np.nan  
    
    # 处理列数据
    processed_values = {}
    for col in ts.columns:
        if col.startswith('LAB_Cate_'):  
            # 取最大值
            processed_values[col] = ts[col].max()
        elif col.startswith('LAB_Str_'):  
            # 取除 0 外出现次数最多的值
            non_zero_values = ts[col][ts[col] != '0']
            if not non_zero_values.empty:
                processed_values[col] = non_zero_values.mode().iloc[0]
            else:
                processed_values[col] = 0
        else:
            # 其他列取中位数
            processed_values[col] = ts[col].median()
    
    # 将结果转为 DataFrame
    result = pd.DataFrame([processed_values])
    
    return result

In [3]:
# LAB
# 24
all_ts = pd.DataFrame()

# 遍历每个 ICUSTAY_ID
for i in tqdm(os.listdir('D:/2024BI_Journal/MIMIC_IV/Final_Lab_2/')):
    try:
        # 读取 dynamic.csv 文件
        file_path = f'D:/2024BI_Journal/MIMIC_IV/Final_Lab_2/{i}'
        ts = pd.read_csv(file_path)
        
        if ts.shape[0]>0:
            ts = ts.iloc[:96,:]

            # 调用 ts_process 函数进行数据处理
            ts = median_lab_process(ts)

            ts['ICUSTAY_ID'] = i.split('.')[0]

            # 合并处理后的数据
            all_ts = pd.concat([all_ts, ts], ignore_index=True)
        
    except Exception as e:
        # 打印错误信息并跳过该文件
        print(f"Error processing file {i}: {e}")
    
all_ts = all_ts.dropna(axis=1, how='all')

  9%|██████▉                                                                      | 6732/74331 [01:55<17:54, 62.93it/s]

Error processing file 30894721.csv: No columns to parse from file


 24%|██████████████████▍                                                         | 18007/74331 [05:25<16:50, 55.76it/s]

Error processing file 32419796.csv: No columns to parse from file


 33%|█████████████████████████▎                                                  | 24706/74331 [08:00<17:55, 46.16it/s]

...

 68%|███████████████████████████████████████████████████▎                        | 50220/74331 [18:14<08:44, 45.93it/s]

Error processing file 36726295.csv: No columns to parse from file


 71%|█████████████████████████████████████████████████████▉                      | 52787/74331 [19:17<08:31, 42.10it/s]

Error processing file 37077064.csv: No columns to parse from file


100%|████████████████████████████████████████████████████████████████████████████| 74331/74331 [28:35<00:00, 43.34it/s]


In [4]:
all_ts.shape

(74321, 11)

In [6]:
all_ts.head(2)

,LAB_50802,LAB_51249,LAB_51277,LAB_51279,LAB_50811,ICUSTAY_ID,LAB_52172,LAB_51493,LAB_51516,LAB_Str_51463,LAB_51300
0,-4.0,34.1,13.5,3.30,10.9,30000153,NaN,NaN,NaN,NaN,NaN
1,1.0,31.0,16.5,2.84,NaN,30000213,49.8,2.0,1.0,NaN,NaN


In [5]:
all_ts.to_csv('D:/2024BI_Journal/MIMIC_IV/Median_B/A24_Lab_median_max_nan.csv',index=False)